# Own LoRA24 plain-head control
Add the exported own_lora24 Dataset, competition data and the same Qwen3-8B Base.
This predicts with the independently trained plain head; it is NOT the original .604 ensemble.
Set a CUDA GPU. Save & Run All must regenerate submission.parquet on the hidden test.


In [ ]:
import json, os, sys, subprocess
from pathlib import Path
ROOT = Path("/kaggle/working/ntl_handoff")
ROOT.mkdir(exist_ok=True)
BUNDLE = {'ntl/__init__.py': '"""Train and validate an independently fitted Name That LLM checkpoint."""\n', 'ntl/checkpoint.py': '"""Validate exported checkpoints without importing torch or executing pickle."""\nimport json\nimport struct\nfrom pathlib import Path\nimport numpy as np\nfrom ntl.common import LABELS, POOL_LAYERS, sha256\n\ndef check_checkpoint(directory, allow_smoke=False):\n    root = Path(directory).resolve()\n    marker = root / "COMPLETE.json"\n    if not marker.exists() and allow_smoke:\n        marker = root / "SMOKE_ONLY.json"\n    if not marker.is_file():\n        raise ValueError("Missing completion marker; interrupted/smoke outputs are not production models")\n    manifest = json.loads(marker.read_text())\n    files = manifest["files"]\n    required = ["adapter/adapter_config.json", "adapter/adapter_model.safetensors",\n                "tokenizer/tokenizer_config.json", "heads.safetensors", "readouts.npz",\n                "anchor_features.npz", "validation_features.npz", "base_config/config.json",\n                "training_meta.json", "training_report.json"]\n    for name in required:\n        if name not in files:\n            raise ValueError(f"Required file absent from manifest: {name}")\n    for name, expected in files.items():\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.is_file():\n            raise ValueError(f"Invalid or missing manifest path: {name}")\n        if sha256(path) != expected:\n            raise ValueError(f"Hash mismatch: {name}")\n    actual = {str(p.relative_to(root)).replace(\'\\\\\', \'/\') for p in root.rglob(\'*\') if p.is_file()}\n    expected_files = {str(Path(k)).replace(\'\\\\\', \'/\') for k in files} | {marker.name}\n    if actual != expected_files:\n        raise ValueError("Unmanifested files or conflicting completion markers found")\n    meta = json.loads((root / "training_meta.json").read_text())\n    report = json.loads((root / "training_report.json").read_text())\n    if meta["labels"] != LABELS or meta["config"]["pool_layers"] != POOL_LAYERS:\n        raise ValueError("Label order or pooled layers differ from the original reader contract")\n    if meta["config"]["keep_layers"] != 24:\n        raise ValueError("Expected 24-layer trunk")\n    if meta["train_sha256"] != manifest["train_sha256"] or meta["job"] != manifest["job"]:\n        raise ValueError("Manifest and metadata disagree")\n    if marker.name == "COMPLETE.json" and (meta["smoke_only"] or report["status"] != "COMPLETE"):\n        raise ValueError("Smoke/error output disguised as a complete model")\n    if marker.name == "COMPLETE.json" and report["successful_updates"] != report["planned_updates"]:\n        raise ValueError("Full training did not finish every planned update")\n    adapter = json.loads((root / "adapter/adapter_config.json").read_text())\n    if adapter.get("peft_type") != "LORA" or adapter.get("r") != meta["config"]["lora_rank"]:\n        raise ValueError("Adapter rank/type differs from metadata")\n    with np.load(root / "readouts.npz", allow_pickle=False) as ro:\n        if ro["classes"].tolist() != LABELS or ro["pool_layers"].tolist() != POOL_LAYERS:\n            raise ValueError("Readout order mismatch")\n        for name, shape in {"plain_coef": (18, 32768), "center_coef": (18, 32768),\n                            "plain_intercept": (18,), "center_intercept": (18,)}.items():\n            if ro[name].shape != shape or not np.isfinite(ro[name]).all():\n                raise ValueError(f"Invalid readout: {name}")\n    with (root / "heads.safetensors").open(\'rb\') as stream:\n        size = struct.unpack(\'<Q\', stream.read(8))[0]\n        if size > 16 * 1024 * 1024:\n            raise ValueError("Invalid safetensors header")\n        header = json.loads(stream.read(size))\n    for i in range(8):\n        for field in ("weight", "bias"):\n            if header.get(f"norms.{i}.{field}", {}).get("shape") != [4096]:\n                raise ValueError("Missing or invalid per-pool LayerNorm")\n    return meta\n', 'ntl/common.py': '"""CPU-safe contracts shared by preparation, training and inference."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nLABELS = sorted([\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",\n    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",\n    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",\n    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",\n])\nPOOL_LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    tmp.replace(path)\n\ndef read_train(path):\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    required = ["id", "prompt_id", "text", "model"]\n    if not set(required) <= set(frame):\n        raise ValueError(f"train.csv must contain {required}")\n    if not len(frame) or not frame.id.is_unique or (frame[required] == "").any().any():\n        raise ValueError("Training IDs must be unique; required fields must be nonempty")\n    if sorted(frame.model.unique()) != LABELS:\n        raise ValueError("Training labels differ from the frozen 18-label contract")\n    return frame\n\ndef make_plan(frame, seed=42, folds=5):\n    groups = np.asarray(sorted(frame.prompt_id.unique()), dtype=str)\n    if len(groups) < folds:\n        raise ValueError("Not enough prompts for the requested number of folds")\n    groups = np.random.default_rng(seed).permutation(groups)\n    jobs = {}\n    for index, held_out in enumerate(np.array_split(groups, folds)):\n        mask = frame.prompt_id.isin(held_out).to_numpy()\n        fit, valid = np.flatnonzero(~mask), np.flatnonzero(mask)\n        if sorted(frame.iloc[fit].model.unique()) != LABELS:\n            raise ValueError(f"Fold {index} fitting side misses classes")\n        jobs[f"fold{index}"] = {"fit": fit.tolist(), "valid": valid.tolist()}\n    jobs["final"] = {"fit": list(range(len(frame))), "valid": []}\n    return {"seed": seed, "folds": folds, "labels": LABELS, "jobs": jobs,\n            "split_method": "shuffle unique prompt IDs then array_split; all answers stay together"}\n\ndef epoch_batches(fit, groups, rng, accumulation=6):\n    """Each microbatch is three distinct answers to one fitting-side prompt."""\n    fit, groups = np.asarray(fit, dtype=int), np.asarray(groups)\n    batches = []\n    for group in np.unique(groups[fit]):\n        members = rng.permutation(fit[groups[fit] == group])\n        if len(members) < 3:\n            raise ValueError(f"Prompt {group!r} has fewer than three fitting answers")\n        for start in range(0, len(members), 3):\n            batch = members[start:start + 3]\n            if len(batch) < 3:\n                batch = np.concatenate([batch, rng.choice(\n                    np.setdiff1d(members, batch), 3 - len(batch), replace=False)])\n            batches.append(batch)\n    batches = [batches[i] for i in rng.permutation(len(batches))]\n    original = len(batches)\n    while len(batches) % accumulation:\n        batches.append(batches[int(rng.integers(original))].copy())\n    return batches, {"original_microbatches": original, "padded_microbatches": len(batches) - original,\n                     "example_exposures": len(batches) * 3, "unique_fit_rows": len(fit)}\n\ndef softmax(values):\n    values = np.asarray(values, dtype=np.float64)\n    exp = np.exp(values - values.max(axis=1, keepdims=True))\n    return exp / exp.sum(axis=1, keepdims=True)\n\ndef balanced_accuracy(y, pred):\n    return float(np.mean([np.mean(pred[y == k] == k) for k in np.unique(y)]))\n', 'ntl/gpu.py': '"""GPU-only model construction. Import this module only on the training machine."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\ndef require_cuda():\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required. CPU tests do not train this model.")\n\ndef load_base(base_path, config):\n    require_cuda()\n    cfg = AutoConfig.from_pretrained(base_path, local_files_only=True)\n    if (cfg.model_type, cfg.hidden_size, cfg.num_hidden_layers) != ("qwen3", 4096, 36):\n        raise ValueError("Use the official 36-layer Qwen3-8B Base, not an instruct model")\n    cfg.num_hidden_layers = config["keep_layers"]\n    cfg.use_cache = False\n    if getattr(cfg, "layer_types", None) is not None:\n        cfg.layer_types = cfg.layer_types[:cfg.num_hidden_layers]\n    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16}[config["dtype"]]\n    model = AutoModel.from_pretrained(\n        base_path, config=cfg, dtype=dtype, local_files_only=True,\n        low_cpu_mem_usage=True, device_map={"": 0}, attn_implementation="sdpa")\n    return model\n\ndef tokenizer_at(path):\n    tok = AutoTokenizer.from_pretrained(path, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.eos_token_id is None:\n        raise ValueError("Tokenizer has no EOS token")\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\ndef tokenize(tok, texts, config):\n    ids = tok(list(texts), add_special_tokens=False, truncation=True,\n              max_length=config["max_text_tokens"])["input_ids"]\n    ids = [part + [tok.eos_token_id] for part in ids]\n    # A hidden passage may be empty: EOS alone is valid. The pool denominator\n    # is clamped in representation, matching the original reader\'s fallback.\n    return ids\n\ndef representation(model, norms, tok, token_rows, layers, dtype):\n    device = model.get_input_embeddings().weight.device\n    batch = tok.pad({"input_ids": token_rows}, return_tensors="pt").to(device)\n    with torch.autocast("cuda", dtype=dtype):\n        hs = model(**batch, output_hidden_states=True, use_cache=False).hidden_states\n    mask = batch["attention_mask"].unsqueeze(-1).float()\n    mask[:, 0] = 0\n    # Match the original inference contract: drop the first token, mean including EOS,\n    # apply an independent FP32 LayerNorm to each pool, then concatenate.\n    with torch.autocast("cuda", enabled=False):\n        z = torch.cat([norm((hs[layer].float() * mask).sum(1) / mask.sum(1).clamp_min(1))\n                       for layer, norm in zip(layers, norms)], dim=-1)\n    if not torch.isfinite(z).all():\n        raise FloatingPointError("Nonfinite feature values")\n    return z\n\ndef extract(model, norms, tok, tokens, rows, config, batch_size=3):\n    rows = np.asarray(rows, dtype=int)\n    out = np.empty((len(rows), 4096 * len(config["pool_layers"])), dtype=np.float32)\n    order = np.argsort([len(tokens[i]) for i in rows], kind="stable")\n    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16}[config["dtype"]]\n    with torch.inference_mode():\n        for start in range(0, len(order), batch_size):\n            positions = order[start:start + batch_size]\n            z = representation(model, norms, tok, [tokens[rows[i]] for i in positions],\n                               config["pool_layers"], dtype)\n            out[positions] = z.cpu().numpy()\n    return out\n\ndef reload_checkpoint(base_path, checkpoint):\n    from peft import PeftModel\n    from safetensors.torch import load_file\n    checkpoint = Path(checkpoint)\n    meta = json.loads((checkpoint / "training_meta.json").read_text())\n    base = load_base(base_path, meta["config"])\n    model = PeftModel.from_pretrained(base, checkpoint / "adapter").eval()\n    norms = nn.ModuleList([nn.LayerNorm(4096) for _ in meta["config"]["pool_layers"]]).cuda().float()\n    state = load_file(str(checkpoint / "heads.safetensors"))\n    norms.load_state_dict({k[len("norms."):]: v for k, v in state.items() if k.startswith("norms.")})\n    norms.eval()\n    return model, norms, tokenizer_at(checkpoint / "tokenizer"), meta\n', 'scripts/bundle_checkpoint.py': '"""Create a weights-only zip for private transfer / Kaggle Dataset upload."""\nimport argparse\nimport sys\nimport zipfile\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.checkpoint import check_checkpoint\nfrom ntl.common import sha256, write_json\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--checkpoint\', required=True)\n    p.add_argument(\'--output\', required=True)\n    args = p.parse_args()\n    ck = Path(args.checkpoint)\n    meta = check_checkpoint(ck)\n    if meta[\'job\'] != \'final\':\n        raise ValueError(\'Package the full-data final checkpoint\')\n    out = Path(args.output)\n    if out.exists() or out.resolve().is_relative_to(ck.resolve()):\n        raise ValueError(\'Choose a fresh zip path outside the checkpoint directory\')\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(out, \'w\', compression=zipfile.ZIP_STORED) as archive:\n        for path in sorted(ck.rglob(\'*\')):\n            if path.is_file():\n                archive.write(path, \'own_lora24/final/\' + path.relative_to(ck).as_posix())\n    write_json(str(out) + \'.sha256.json\', {\'sha256\': sha256(out),\n        \'layout\': \'own_lora24/final\', \'identity\': meta[\'identity\']})\n    print(f\'Weights transfer archive (do not commit to GitHub): {out}\')\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/check_parity.py': '"""Reload a saved adapter and reproduce its first three training features."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.common import read_train, sha256, softmax, write_json\nfrom ntl.checkpoint import check_checkpoint\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--checkpoint\', required=True)\n    p.add_argument(\'--base-model\', required=True)\n    p.add_argument(\'--train-csv\', required=True)\n    p.add_argument(\'--allow-smoke\', action=\'store_true\')\n    p.add_argument(\'--output\', required=True, help=\'Outside checkpoint directory (manifest must stay immutable)\')\n    args = p.parse_args()\n    ck = Path(args.checkpoint)\n    if Path(args.output).resolve().is_relative_to(ck.resolve()):\n        raise ValueError(\'Write parity report outside the checkpoint directory\')\n    meta = check_checkpoint(ck, args.allow_smoke)\n    if sha256(args.train_csv) != meta[\'train_sha256\']:\n        raise ValueError(\'Training rows differ from export\')\n    if sha256(Path(args.base_model) / \'config.json\') != meta[\'base_config_sha256\']:\n        raise ValueError(\'Base configuration differs from export\')\n    import numpy as np\n    from ntl.gpu import reload_checkpoint, tokenize, extract\n    frame = read_train(args.train_csv)\n    model, norms, tok, _ = reload_checkpoint(args.base_model, ck)\n    with np.load(ck / \'anchor_features.npz\', allow_pickle=False) as saved:\n        ids = saved[\'ids\']; expected = saved[\'features\']\n    positions = frame.set_index(\'id\').index.get_indexer(ids)\n    if (positions < 0).any():\n        raise ValueError(\'Anchor IDs not found\')\n    tokens = tokenize(tok, frame.text.iloc[positions].tolist(), meta[\'config\'])\n    actual = extract(model, norms, tok, tokens, np.arange(len(tokens)), meta[\'config\'])\n    rel = float(np.abs(actual - expected).mean() / (np.abs(expected).mean() + 1e-9))\n    with np.load(ck / \'readouts.npz\', allow_pickle=False) as ro:\n        prob = softmax(actual @ ro[\'plain_coef\'].T + ro[\'plain_intercept\'])\n    passed = bool(np.isfinite(actual).all() and rel < .02 and np.isfinite(prob).all()\n                  and np.allclose(prob.sum(1), 1) and np.all((prob >= 0) & (prob <= 1)))\n    result = {\'passed\': passed, \'mean_relative_feature_error\': rel,\n              \'max_absolute_feature_error\': float(np.abs(actual - expected).max()),\n              \'threshold\': .02, \'train_sha256\': meta[\'train_sha256\'], \'job\': meta[\'job\'],\n              \'smoke_only\': meta[\'smoke_only\'], \'checkpoint_marker_sha256\': sha256(\n                  ck / (\'SMOKE_ONLY.json\' if meta[\'smoke_only\'] else \'COMPLETE.json\')),\n              \'probability_shape\': list(prob.shape)}\n    write_json(args.output, result)\n    print(json.dumps(result, indent=2))\n    if not passed:\n        raise SystemExit(1)\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/evaluate_oof.py': '"""Evaluate prompt-disjoint plain-head predictions; never fit on validation labels."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.checkpoint import check_checkpoint\nfrom ntl.common import LABELS, balanced_accuracy, read_train, sha256, softmax, write_json\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--plan\', required=True)\n    p.add_argument(\'--checkpoints\', required=True, help=\'Parent directory containing fold0..fold4\')\n    p.add_argument(\'--output\', default=\'artifacts/oof_report.json\')\n    args = p.parse_args()\n    import numpy as np\n    plan = json.loads(Path(args.plan).read_text())\n    if sha256(plan[\'train_csv\']) != plan[\'train_sha256\']:\n        raise ValueError(\'Changed train data\')\n    frame = read_train(plan[\'train_csv\'])\n    pred = np.empty(len(frame), int); coverage = np.zeros(len(frame), int); audits = []\n    plan_hash = sha256(args.plan)\n    for name, indices in plan[\'jobs\'].items():\n        if name == \'final\':\n            continue\n        ck = Path(args.checkpoints) / name\n        meta = check_checkpoint(ck)\n        if meta[\'job\'] != name or meta[\'plan_sha256\'] != plan_hash:\n            raise ValueError(\'Checkpoint was fitted with a different plan/job\')\n        rows = np.asarray(indices[\'valid\'], int)\n        with np.load(ck / \'validation_features.npz\', allow_pickle=False) as bank:\n            if not np.array_equal(bank[\'rows\'], rows) or bank[\'ids\'].tolist() != frame.id.iloc[rows].tolist():\n                raise ValueError(\'Validation row identity/order mismatch\')\n            features = bank[\'features\']\n        with np.load(ck / \'readouts.npz\', allow_pickle=False) as ro:\n            logits = features @ ro[\'plain_coef\'].T + ro[\'plain_intercept\']\n        if not np.isfinite(logits).all():\n            raise ValueError(\'Nonfinite held-out scores\')\n        pred[rows] = logits.argmax(1); coverage[rows] += 1\n        y = np.searchsorted(LABELS, frame.model.iloc[rows].to_numpy())\n        audits.append({\'job\': name, \'rows\': len(rows), \'plain_balanced_accuracy\': balanced_accuracy(y, pred[rows])})\n    if not np.all(coverage == 1):\n        raise ValueError(\'Each training row must be held out exactly once\')\n    y = np.searchsorted(LABELS, frame.model.to_numpy())\n    report = {\'status\': \'COMPLETE\', \'train_sha256\': plan[\'train_sha256\'], \'folds\': audits,\n              \'plain_oof_balanced_accuracy\': balanced_accuracy(y, pred),\n              \'scope\': \'Own plain head only; no centered-head validation or ensemble tuning performed\',\n              \'leaderboard_score\': None}\n    write_json(args.output, report)\n    print(json.dumps(report, indent=2))\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/make_release.py': '"""Build a source-only zip from the scanned release allowlist."""\nimport argparse\nimport json\nimport sys\nimport zipfile\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom scan_release import ROOT, scan\nfrom ntl.common import sha256, write_json\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\', required=True)\n    args = p.parse_args()\n    output = Path(args.output)\n    if output.exists():\n        raise FileExistsError(output)\n    files, problems = scan()\n    if problems:\n        raise ValueError(\'\\n\'.join(problems))\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(output, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in files:\n            archive.write(path, ROOT.name + \'/\' + path.relative_to(ROOT).as_posix())\n    report = {\'file\': output.name, \'files\': len(files), \'bytes\': output.stat().st_size,\n              \'sha256\': sha256(output), \'scope\': \'source/config/docs/code-only notebooks; no data or weights\'}\n    write_json(str(output) + \'.sha256.json\', report)\n    print(json.dumps(report, indent=2))\n', 'scripts/patch_notebook.py': '"""Patch a separately downloaded original notebook; do not execute its source."""\nimport argparse\nimport ast\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.common import sha256, write_json\n\nOLD_MARKER = \'**/lora24/final/COMPLETE.json\'\n\ndef patch_source(source, marker):\n    tree = ast.parse(source)\n    found = 0\n    class Update(ast.NodeTransformer):\n        def visit_Assign(self, node):\n            nonlocal found\n            if any(isinstance(t, ast.Name) and t.id == \'_GAUSSIAN_SOURCE\' for t in node.targets):\n                if not isinstance(node.value, ast.Constant) or not isinstance(node.value.value, str):\n                    raise ValueError(\'_GAUSSIAN_SOURCE is no longer a literal; review the new source\')\n                embedded = node.value.value\n                if embedded.count(OLD_MARKER) != 1 or \'lora24_public\' not in embedded:\n                    raise ValueError(\'Expected original checkpoint configuration not found exactly once\')\n                embedded = embedded.replace(OLD_MARKER, marker).replace(\'lora24_public\', \'lora24_own\')\n                ast.parse(embedded)\n                node.value = ast.copy_location(ast.Constant(embedded), node.value)\n                found += 1\n            return node\n    result = Update().visit(tree)\n    ast.fix_missing_locations(result)\n    return ast.unparse(result) + \'\\n\', found\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--input\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--checkpoint-glob\', default=\'**/own_lora24/final/COMPLETE.json\')\n    args = p.parse_args()\n    if Path(args.output).exists():\n        raise FileExistsError(args.output)\n    if Path(args.input).resolve() == Path(args.output).resolve():\n        raise ValueError(\'Keep the original notebook unchanged\')\n    if args.checkpoint_glob.startswith(\'/\') or \'..\' in Path(args.checkpoint_glob).parts:\n        raise ValueError(\'Use a relative glob under /kaggle/input\')\n    nb = json.loads(Path(args.input).read_text(encoding=\'utf-8\'))\n    matches = 0\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\'] != \'code\':\n            continue\n        source = cell[\'source\']; source = \'\'.join(source) if isinstance(source, list) else source\n        if \'_GAUSSIAN_SOURCE\' in source:\n            source, count = patch_source(source, args.checkpoint_glob)\n            matches += count\n        cell[\'source\'] = source.splitlines(keepends=True)\n        cell[\'outputs\'] = []; cell[\'execution_count\'] = None\n    if matches != 1:\n        raise ValueError(f\'Expected exactly one original stack, got {matches}\')\n    markers = [args.checkpoint_glob,\n        \'**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json\',\n        \'**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json\',\n        \'**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json\',\n        \'**/tokattn_final.pt\', \'**/tokattn_debl23_final.pt\']\n    early = (\'# Fail early on absent inputs; this does not validate model identity or GPU compatibility.\\n\'\n             \'import glob\\n\'\n             f\'_required_markers = {markers!r}\\n\'\n             \'_missing = [p for p in _required_markers if not glob.glob("/kaggle/input/" + p, recursive=True)]\\n\'\n             \'if _missing:\\n    raise FileNotFoundError("Missing inputs: " + repr(_missing))\\n\'\n             \'print("Input marker precheck passed; verify all other models and run GPU parity separately.")\\n\')\n    nb[\'cells\'].insert(0, {\'cell_type\': \'markdown\', \'metadata\': {}, \'source\': [\n        \'# Own retrained LoRA replacement\\n\',\n        \'One original member is replaced by an independently trained checkpoint. This is a new unscored experiment.\\n\',\n        \'Source: https://www.kaggle.com/code/kiza123123/ntl-0604-dual-lora-gaussian-hungarian?scriptVersionId=360450415\\n\']})\n    nb[\'cells\'].insert(1, {\'cell_type\': \'code\', \'metadata\': {}, \'source\': early.splitlines(keepends=True),\n                            \'outputs\': [], \'execution_count\': None})\n    nb[\'metadata\'].pop(\'widgets\', None)\n    Path(args.output).parent.mkdir(parents=True, exist_ok=True)\n    write_json(args.output, nb)\n    write_json(str(args.output) + \'.patch.json\', {\n        \'input_sha256\': sha256(args.input), \'output_sha256\': sha256(args.output),\n        \'replaced_member\': \'lora24_public -> lora24_own\', \'new_marker\': args.checkpoint_glob,\n        \'other_changes\': \'Outputs cleared; early file checks; ast.unparse formatting; other model logic preserved\',\n        \'validation\': \'Python syntax only; full GPU ensemble not run\'})\n    print(f\'Wrote new unscored notebook: {args.output}\')\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/predict.py': '"""Standalone plain-head control. This is NOT the original multi-model ensemble."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.checkpoint import check_checkpoint\nfrom ntl.common import LABELS, sha256, softmax, write_json\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--checkpoint\', required=True)\n    p.add_argument(\'--base-model\', required=True)\n    p.add_argument(\'--test-csv\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--batch-size\', type=int, default=3)\n    args = p.parse_args()\n    import numpy as np\n    import pandas as pd\n    meta = check_checkpoint(args.checkpoint)\n    if meta[\'job\'] != \'final\':\n        raise ValueError(\'Use the full-data final checkpoint for submission\')\n    if sha256(Path(args.base_model) / \'config.json\') != meta[\'base_config_sha256\']:\n        raise ValueError(\'Wrong base configuration\')\n    from ntl.gpu import reload_checkpoint, tokenize, extract\n    if Path(args.output).exists():\n        raise FileExistsError(args.output)\n    test = pd.read_csv(args.test_csv, dtype=str, keep_default_na=False)\n    if not {\'id\', \'text\'} <= set(test) or not len(test) or not test.id.is_unique:\n        raise ValueError(\'Invalid test ID/text schema\')\n    # Empty hidden-test passages remain defined; whitespace produces a real text token.\n    texts = test.text.map(lambda s: s if s.strip() else \' \').tolist()\n    model, norms, tok, _ = reload_checkpoint(args.base_model, args.checkpoint)\n    tokens = tokenize(tok, texts, meta[\'config\'])\n    z = extract(model, norms, tok, tokens, np.arange(len(tokens)), meta[\'config\'], args.batch_size)\n    with np.load(Path(args.checkpoint) / \'readouts.npz\', allow_pickle=False) as ro:\n        prob = softmax(z @ ro[\'plain_coef\'].T + ro[\'plain_intercept\'])\n    if not np.isfinite(prob).all() or not np.allclose(prob.sum(1), 1):\n        raise ValueError(\'Invalid probabilities\')\n    result = pd.DataFrame({\'id\': test.id, \'model\': np.asarray(LABELS)[prob.argmax(1)]})\n    out = Path(args.output); out.parent.mkdir(parents=True, exist_ok=True)\n    result.to_parquet(out, index=False)\n    write_json(out.with_suffix(\'.audit.json\'), {\'rows\': len(result), \'sha256\': sha256(out),\n        \'checkpoint_marker_sha256\': sha256(Path(args.checkpoint) / \'COMPLETE.json\'),\n        \'prediction_counts\': result.model.value_counts().to_dict(),\n        \'identity\': \'own plain LoRA control; no original ensemble score claim\'})\n    print(f\'Wrote {len(result)} rows: {out}\')\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/preflight.py': '"""Check hardware, imports and input contracts before spending GPU time."""\nimport argparse\nimport importlib.metadata\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.common import sha256, write_json\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--plan", required=True)\n    parser.add_argument("--output", default="artifacts/preflight.json")\n    args = parser.parse_args()\n    plan = json.loads(Path(args.plan).read_text())\n    report = {"python": sys.version, "passed": False, "packages": {}}\n    try:\n        for name in ["torch", "transformers", "peft", "accelerate", "safetensors", "numpy", "pandas"]:\n            report["packages"][name] = importlib.metadata.version(name)\n        import torch\n        import peft\n        import transformers\n        assert torch.cuda.is_available(), "CUDA unavailable; this step needs a GPU machine"\n        report["gpus"] = [{"name": torch.cuda.get_device_name(i),\n                           "total_bytes": torch.cuda.get_device_properties(i).total_memory,\n                           "free_bytes": torch.cuda.mem_get_info(i)[0]}\n                          for i in range(torch.cuda.device_count())]\n        assert sha256(plan["train_csv"]) == plan["train_sha256"], "Training CSV changed"\n        assert sha256(Path(plan["base_model"]) / "config.json") == plan["base_config_sha256"]\n        report["plan_sha256"] = sha256(args.plan)\n        report["passed"] = True\n    except Exception as error:\n        report["error"] = f"{type(error).__name__}: {error}"\n    write_json(args.output, report)\n    print(json.dumps(report, indent=2))\n    if not report["passed"]:\n        raise SystemExit(1)\n\nif __name__ == "__main__":\n    main()\n', 'scripts/prepare.py': '"""Create a prompt-disjoint training plan from labeled training data."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.common import make_plan, read_train, sha256, write_json\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--train-csv", required=True)\n    p.add_argument("--base-model", required=True, help="Local complete Qwen3-8B Base directory")\n    p.add_argument("--config", default="configs/train.json")\n    p.add_argument("--output", default="artifacts/plan.json")\n    args = p.parse_args()\n    if Path(args.output).exists():\n        raise FileExistsError(f"Choose a fresh plan path: {args.output}")\n    config = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    base = Path(args.base_model).resolve()\n    model_config = json.loads((base / "config.json").read_text(encoding="utf-8"))\n    expected = {"model_type": "qwen3", "hidden_size": 4096, "num_hidden_layers": 36}\n    if any(model_config.get(k) != v for k, v in expected.items()):\n        raise ValueError(f"Expected unmodified Qwen3-8B Base: {expected}")\n    index = base / "model.safetensors.index.json"\n    if index.exists():\n        shards = set(json.loads(index.read_text())["weight_map"].values())\n        if not all((base / name).is_file() for name in shards):\n            raise ValueError("Some indexed model shards are missing")\n    elif not (base / "model.safetensors").exists():\n        raise ValueError("Missing model.safetensors or its shard index")\n    if not (base / "tokenizer_config.json").is_file():\n        raise ValueError("Missing tokenizer_config.json")\n    frame = read_train(args.train_csv)\n    plan = make_plan(frame, config["seed"], config["folds"])\n    plan.update(train_csv=str(Path(args.train_csv).resolve()), train_sha256=sha256(args.train_csv),\n                train_rows=len(frame), train_prompts=int(frame.prompt_id.nunique()),\n                base_model=str(base), base_config_sha256=sha256(base / "config.json"), config=config,\n                identity="independent retraining; no original-weight or score parity claim")\n    write_json(args.output, plan)\n    print(f"Prepared {len(frame)} rows / {frame.prompt_id.nunique()} prompts -> {args.output}")\n    print("Jobs:", ", ".join(plan["jobs"]))\n\nif __name__ == "__main__":\n    main()\n', 'scripts/run_jobs.py': '"""Schedule independent fold/final jobs, one model per visible GPU."""\nimport argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.common import sha256\nfrom ntl.checkpoint import check_checkpoint\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--plan\', required=True)\n    p.add_argument(\'--output-root\', required=True)\n    p.add_argument(\'--devices\', default=\'0\', help=\'Indices within the current CUDA_VISIBLE_DEVICES namespace\')\n    p.add_argument(\'--jobs\', default=\'fold0,fold1,fold2,fold3,fold4,final\')\n    p.add_argument(\'--skip-complete\', action=\'store_true\', help=\'Verify matching complete checkpoints before skipping\')\n    args = p.parse_args()\n    plan = json.loads(Path(args.plan).read_text())\n    devices = [d.strip() for d in args.devices.split(\',\')]\n    if not devices or len(set(devices)) != len(devices) or not all(d.isdigit() for d in devices):\n        p.error(\'Provide unique nonnegative GPU indices\')\n    outer = os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n    mapping = outer.split(\',\') if outer is not None else None\n    if mapping is not None:\n        if any(int(d) >= len(mapping) for d in devices):\n            p.error(\'Requested device is outside CUDA_VISIBLE_DEVICES\')\n        devices = [mapping[int(d)].strip() for d in devices]\n    jobs = [j.strip() for j in args.jobs.split(\',\')]\n    if len(set(jobs)) != len(jobs) or any(j not in plan[\'jobs\'] for j in jobs):\n        p.error(\'Unknown or duplicate jobs\')\n    root = Path(args.output_root).resolve(); root.mkdir(parents=True, exist_ok=True)\n    pending = []\n    for job in jobs:\n        ck = root / job\n        if ck.exists():\n            if not args.skip_complete:\n                raise FileExistsError(f\'{ck}: use a fresh output root, or explicitly verify/skip complete jobs\')\n            meta = check_checkpoint(ck)\n            if meta[\'plan_sha256\'] != sha256(args.plan) or meta[\'job\'] != job:\n                raise ValueError(\'Existing checkpoint belongs to a different plan/job\')\n            print(f\'Verified existing checkpoint: {job}\', flush=True)\n        else:\n            pending.append(job)\n    running = {}\n    try:\n        while pending or running:\n            for device in devices:\n                if device in running or not pending:\n                    continue\n                job = pending.pop(0)\n                log = (root / f\'{job}.log\').open(\'w\', encoding=\'utf-8\')\n                env = os.environ.copy(); env[\'CUDA_VISIBLE_DEVICES\'] = device\n                command = [sys.executable, str(Path(__file__).with_name(\'train.py\')),\n                           \'--plan\', str(Path(args.plan).resolve()), \'--job\', job, \'--output\', str(root / job)]\n                proc = subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT)\n                running[device] = (proc, log, job)\n                print(f\'Started {job} on visible GPU {device}: {root / (job + ".log")}\', flush=True)\n            for device, (proc, log, job) in list(running.items()):\n                if proc.poll() is None:\n                    continue\n                log.close(); del running[device]\n                if proc.returncode:\n                    tail = (root / f\'{job}.log\').read_text(errors=\'replace\')[-5000:]\n                    raise RuntimeError(f\'{job} failed ({proc.returncode}):\\n{tail}\')\n                check_checkpoint(root / job)\n                print(f\'Complete: {job}\', flush=True)\n            if running:\n                time.sleep(2)\n    finally:\n        for proc, log, job in running.values():\n            if proc.poll() is None:\n                proc.terminate()\n                try:\n                    proc.wait(timeout=20)\n                except subprocess.TimeoutExpired:\n                    proc.kill(); proc.wait()\n            log.close()\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/scan_release.py': '"""Scan only the explicit public-source release allowlist; no GPU required."""\nimport json\nimport re\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nROOT_FILES = {\'README.md\', \'AGENTS.md\', \'AI_START_HERE.md\', \'LICENSE\', \'.gitignore\', \'.gitattributes\',\n              \'requirements-cpu.txt\', \'requirements-gpu.txt\'}\nPUBLIC_DIRS = {\'ntl\', \'scripts\', \'configs\', \'docs\', \'notebooks\', \'tests\', \'evidence\', \'.github\'}\nSECRET_PATTERNS = [r\'ghp_[A-Za-z0-9]{30,}\', r\'github_pat_[A-Za-z0-9_]{30,}\',\n                   r\'hf_[A-Za-z0-9]{25,}\', r\'-----BEGIN (?:RSA |OPENSSH |EC )?PRIVATE KEY-----\']\nFORBIDDEN_SUFFIX = {\'.csv\', \'.tsv\', \'.parquet\', \'.npy\', \'.npz\', \'.safetensors\', \'.pt\', \'.pth\', \'.bin\', \'.zip\', \'.log\'}\n\ndef release_files(root=ROOT):\n    for path in sorted(root.rglob(\'*\')):\n        if not path.is_file() or \'__pycache__\' in path.parts or path.suffix == \'.pyc\':\n            continue\n        rel = path.relative_to(root)\n        if (len(rel.parts) == 1 and path.name in ROOT_FILES) or rel.parts[0] in PUBLIC_DIRS:\n            yield path\n\ndef scan(root=ROOT):\n    problems = []\n    files = list(release_files(root))\n    for path in files:\n        rel = path.relative_to(root)\n        if path.is_symlink():\n            problems.append(f\'{rel}: symlink not allowed\')\n        if path.suffix.lower() in FORBIDDEN_SUFFIX or path.name.lower() in {\'kaggle.json\', \'.env\'}:\n            problems.append(f\'{rel}: data/weight/credential artifact\')\n        if path.stat().st_size > 2 * 1024 * 1024:\n            problems.append(f\'{rel}: exceeds 2 MiB public source limit\')\n        try:\n            text = path.read_text(encoding=\'utf-8\')\n        except UnicodeError:\n            problems.append(f\'{rel}: non-text file in source release\')\n            continue\n        for pattern in SECRET_PATTERNS:\n            if re.search(pattern, text):\n                problems.append(f\'{rel}: possible credential; inspect without printing matched data\')\n        if path.suffix == \'.ipynb\':\n            try:\n                nb = json.loads(text)\n                for cell in nb[\'cells\']:\n                    if cell.get(\'outputs\') or cell.get(\'execution_count\') is not None:\n                        problems.append(f\'{rel}: saved notebook execution data\')\n            except (ValueError, KeyError):\n                problems.append(f\'{rel}: malformed notebook\')\n    return files, problems\n\nif __name__ == \'__main__\':\n    files, problems = scan()\n    print(json.dumps({\'status\': \'FAIL\' if problems else \'PASS\', \'public_files\': len(files),\n                      \'problems\': problems, \'scope\': \'Explicit source allowlist; runtime dirs excluded\'}, indent=2))\n    raise SystemExit(bool(problems))\n', 'scripts/train.py': '"""Train one prompt-disjoint fold or final model on one visible CUDA GPU."""\nimport argparse\nimport gc\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.common import LABELS, POOL_LAYERS, epoch_batches, read_train, sha256, write_json\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--plan", required=True)\n    p.add_argument("--job", required=True, help="fold0..fold4 or final")\n    p.add_argument("--output", required=True, help="Fresh checkpoint directory; never overwrites")\n    p.add_argument("--max-updates", type=int, help="Smoke only: no COMPLETE.json is written")\n    args = p.parse_args()\n    if args.max_updates is not None and args.max_updates < 1:\n        p.error("--max-updates must be positive")\n    out = Path(args.output)\n    if out.exists():\n        raise FileExistsError(f"Choose a new output directory, or archive the old run yourself: {out}")\n    plan = json.loads(Path(args.plan).read_text())\n    config = plan["config"]\n    if config["epochs"] < 1 or config["accumulation"] < 1:\n        raise ValueError("epochs and accumulation must be positive")\n    if config["pool_layers"] != POOL_LAYERS or config["keep_layers"] != 24 or config["microbatch"] != 3:\n        raise ValueError("This implementation fixes the original 24-layer / 8-pool / triple contract")\n    if sha256(plan["train_csv"]) != plan["train_sha256"]:\n        raise ValueError("Training data changed after preparing the plan")\n    if sha256(Path(plan["base_model"]) / "config.json") != plan["base_config_sha256"]:\n        raise ValueError("Base configuration changed after preparing the plan")\n    frame = read_train(plan["train_csv"])\n    import numpy as np\n    import torch\n    from torch import nn\n    from peft import LoraConfig, get_peft_model\n    from safetensors.torch import save_file\n    from ntl.gpu import extract, load_base, representation, require_cuda, tokenize, tokenizer_at\n    require_cuda()\n    seed = config["seed"]\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(2)\n    fit = np.asarray(plan["jobs"][args.job]["fit"], dtype=int)\n    valid = np.asarray(plan["jobs"][args.job]["valid"], dtype=int)\n    groups = frame.prompt_id.to_numpy()\n    if set(groups[fit]) & set(groups[valid]) or set(fit) & set(valid):\n        raise ValueError("Prompt or row leakage in plan")\n    if sorted(frame.iloc[fit].model.unique()) != LABELS:\n        raise ValueError("Fitting side must contain every class")\n    if plan["labels"] != LABELS:\n        raise ValueError("Label order changed")\n    target = np.searchsorted(LABELS, frame.model.to_numpy())\n    rng = np.random.default_rng(seed)\n    epochs = [epoch_batches(fit, groups, rng, config["accumulation"])\n              for _ in range(config["epochs"])]\n    total = sum(len(batches) // config["accumulation"] for batches, _ in epochs)\n    limit = min(total, args.max_updates) if args.max_updates else total\n    out.mkdir(parents=True)\n    report = {"status": "STARTING", "job": args.job, "smoke_only": args.max_updates is not None,\n              "train_sha256": plan["train_sha256"], "fit_rows": len(fit), "valid_rows": len(valid),\n              "planned_updates": total, "update_limit": limit,\n              "epoch_sampling": [audit for _, audit in epochs], "updates": [],\n              "gpu": torch.cuda.get_device_name(0), "python": sys.version,\n              "packages": {k: importlib.metadata.version(k) for k in\n                           ["torch", "transformers", "peft", "accelerate", "safetensors", "numpy", "pandas"]}}\n    write_json(out / "training_report.json", report)\n    started = time.monotonic()\n    try:\n        tok = tokenizer_at(plan["base_model"])\n        tokens = tokenize(tok, frame.text.tolist(), config)\n        base = load_base(plan["base_model"], config)\n        base.requires_grad_(False)\n        model = get_peft_model(base, LoraConfig(\n            r=config["lora_rank"], lora_alpha=config["lora_alpha"],\n            lora_dropout=config["lora_dropout"], bias="none",\n            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n        for parameter in model.parameters():\n            if parameter.requires_grad:\n                parameter.data = parameter.data.float()\n        norms = nn.ModuleList([nn.LayerNorm(4096) for _ in POOL_LAYERS]).cuda().float()\n        plain = nn.Linear(4096 * len(POOL_LAYERS), len(LABELS)).cuda().float()\n        centered = nn.Linear(4096 * len(POOL_LAYERS), len(LABELS)).cuda().float()\n        dropout = nn.Dropout(config["head_dropout"])\n        adapter_params = [v for v in model.parameters() if v.requires_grad]\n        head_params = list(norms.parameters()) + list(plain.parameters()) + list(centered.parameters())\n        parameters = adapter_params + head_params\n        if not adapter_params or not all(v.dtype == torch.float32 for v in parameters):\n            raise ValueError("Expected FP32 trainable adapters and heads")\n        report["trainable_parameters"] = sum(v.numel() for v in parameters)\n        optimizer = torch.optim.AdamW([\n            {"params": adapter_params, "lr": config["adapter_lr"]},\n            {"params": head_params, "lr": config["head_lr"]}], weight_decay=config["weight_decay"])\n        warmup = max(1, int(total * config["warmup_fraction"]))\n        def schedule(step):\n            if step < warmup:\n                return (step + 1) / warmup\n            return .5 * (1 + math.cos(math.pi * min(1., (step - warmup) / max(1, total - warmup))))\n        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n        fp16 = config["dtype"] == "float16"\n        dtype = torch.float16 if fp16 else torch.bfloat16\n        scaler = torch.amp.GradScaler("cuda", enabled=fp16, init_scale=256.)\n        model.train(); norms.train(); plain.train(); centered.train()\n        torch.cuda.reset_peak_memory_stats()\n        report["status"] = "TRAINING"\n        for epoch, (batches, _) in enumerate(epochs, start=1):\n            for start in range(0, len(batches), config["accumulation"]):\n                if len(report["updates"]) >= limit:\n                    break\n                chunk = batches[start:start + config["accumulation"]]\n                cpu_rng, cuda_rng = torch.get_rng_state(), torch.cuda.get_rng_state()\n                update_started = time.monotonic()\n                for attempt in range(config["max_overflow_retries"] + 1):\n                    torch.set_rng_state(cpu_rng); torch.cuda.set_rng_state(cuda_rng)\n                    optimizer.zero_grad(set_to_none=True)\n                    losses = []\n                    for indices in chunk:\n                        z = representation(model, norms, tok, [tokens[i] for i in indices], POOL_LAYERS, dtype)\n                        y = torch.as_tensor(target[indices], device="cuda", dtype=torch.long)\n                        with torch.autocast("cuda", enabled=False):\n                            lp = nn.functional.cross_entropy(plain(dropout(z)), y)\n                            lc = nn.functional.cross_entropy(centered(dropout(z - z.mean(0, keepdim=True))), y)\n                            loss = lp + lc\n                        if not torch.isfinite(loss):\n                            raise FloatingPointError("Nonfinite forward loss; no automatic masking")\n                        losses.append(float(loss.detach()))\n                        scaler.scale(loss / len(chunk)).backward()\n                    scaler.unscale_(optimizer)\n                    norm = torch.nn.utils.clip_grad_norm_(parameters, config["gradient_clip"])\n                    finite = bool(torch.isfinite(norm))\n                    if not finite and not fp16:\n                        raise FloatingPointError("Nonfinite BF16 gradients")\n                    if finite:\n                        if not any(v.grad is not None and bool(torch.count_nonzero(v.grad))\n                                   for v in adapter_params):\n                            raise RuntimeError("All adapter gradients are zero; training path is broken")\n                        scaler.step(optimizer)\n                        scaler.update()\n                        scheduler.step()\n                        break\n                    # Never step on a nonfinite total norm, even if individual gradients\n                    # were finite and only the norm reduction overflowed.\n                    scaler.update(new_scale=scaler.get_scale() / 2)\n                    print(json.dumps({"event": "AMP_OVERFLOW", "attempt": attempt + 1,\n                                      "scale": scaler.get_scale()}), flush=True)\n                    if attempt == config["max_overflow_retries"] or scaler.get_scale() < 1:\n                        raise FloatingPointError("Repeated AMP overflow; inspect environment/dtype")\n                torch.cuda.synchronize()\n                record = {"number": len(report["updates"]) + 1, "epoch": epoch,\n                          "loss": float(np.mean(losses)), "gradient_norm": float(norm),\n                          "overflow_retries": attempt, "scale": scaler.get_scale(),\n                          "seconds": time.monotonic() - update_started}\n                report["updates"].append(record)\n                write_json(out / "training_report.json", report)\n                print(json.dumps(record), flush=True)\n            if len(report["updates"]) >= limit:\n                break\n        model.eval(); norms.eval(); plain.eval(); centered.eval()\n        model.gradient_checkpointing_disable()\n        optimizer.zero_grad(set_to_none=True)\n        valid_z = extract(model, norms, tok, tokens, valid, config)\n        anchors = np.arange(min(3, len(frame)))\n        anchor_z = extract(model, norms, tok, tokens, anchors, config)\n        model.save_pretrained(out / "adapter", safe_serialization=True, save_embedding_layers=False)\n        tok.save_pretrained(out / "tokenizer")\n        base.config.save_pretrained(out / "base_config")\n        state = {}\n        for prefix, module in [("norms", norms), ("plain", plain), ("centered", centered)]:\n            state.update({f"{prefix}.{k}": v.detach().cpu().contiguous() for k, v in module.state_dict().items()})\n        save_file(state, str(out / "heads.safetensors"))\n        np.savez_compressed(out / "readouts.npz", classes=np.asarray(LABELS), pool_layers=np.asarray(POOL_LAYERS),\n            plain_coef=plain.weight.detach().cpu().numpy(), plain_intercept=plain.bias.detach().cpu().numpy(),\n            center_coef=centered.weight.detach().cpu().numpy(), center_intercept=centered.bias.detach().cpu().numpy(),\n            train_sha256=plan["train_sha256"])\n        np.savez_compressed(out / "validation_features.npz", ids=frame.id.iloc[valid].to_numpy(dtype=str),\n                            rows=valid, features=valid_z, classes=np.asarray(LABELS))\n        np.savez_compressed(out / "anchor_features.npz", ids=frame.id.iloc[anchors].to_numpy(dtype=str),\n                            features=anchor_z, classes=np.asarray(LABELS))\n        source_root = Path(__file__).resolve().parents[1]\n        source_hashes = {str(p.relative_to(source_root)): sha256(p)\n                         for folder in ["ntl", "scripts"] for p in (source_root / folder).glob("*.py")}\n        write_json(out / "training_meta.json", {\n            "schema": 1, "job": args.job, "experiment": config["experiment"], "config": config,\n            "labels": LABELS, "train_sha256": plan["train_sha256"],\n            "base_config_sha256": plan["base_config_sha256"], "plan_sha256": sha256(args.plan),\n            "source_sha256": source_hashes, "smoke_only": args.max_updates is not None,\n            "identity": "independent retraining; not verified identical to lightningv08 checkpoint"})\n        report.update(status="SMOKE_COMPLETE" if args.max_updates else "COMPLETE",\n                      seconds=time.monotonic() - started,\n                      peak_allocated_bytes=torch.cuda.max_memory_allocated(),\n                      successful_updates=len(report["updates"]))\n        write_json(out / "training_report.json", report)\n        files = {str(p.relative_to(out)): sha256(p) for p in sorted(out.rglob("*")) if p.is_file()}\n        marker = "SMOKE_ONLY.json" if args.max_updates else "COMPLETE.json"\n        write_json(out / marker, {"schema": 1, "job": args.job, "train_sha256": plan["train_sha256"], "files": files})\n        print(f"Exported {marker}: {out}", flush=True)\n    except Exception as error:\n        report.update(status="ERROR", error=f"{type(error).__name__}: {error}")\n        write_json(out / "training_report.json", report)\n        raise\n\nif __name__ == "__main__":\n    main()\n', 'scripts/validate_checkpoint.py': 'import argparse\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom ntl.checkpoint import check_checkpoint\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description="CPU structural/hash check; GPU parity is a separate required step")\n    p.add_argument(\'--checkpoint\', required=True)\n    p.add_argument(\'--allow-smoke\', action=\'store_true\')\n    args = p.parse_args()\n    meta = check_checkpoint(args.checkpoint, args.allow_smoke)\n    print(json.dumps({"structure_and_hashes": "PASS", "job": meta[\'job\'],\n                      "smoke_only": meta[\'smoke_only\'], "gpu_parity": "NOT_CHECKED"}, indent=2))\n', 'configs/train.json': '{\n  "experiment": "own-qwen3-8b-lora24-v1",\n  "seed": 42,\n  "folds": 5,\n  "epochs": 3,\n  "keep_layers": 24,\n  "pool_layers": [2, 4, 6, 8, 12, 16, 20, 24],\n  "max_text_tokens": 447,\n  "microbatch": 3,\n  "accumulation": 6,\n  "lora_rank": 16,\n  "lora_alpha": 32,\n  "lora_dropout": 0.05,\n  "head_dropout": 0.1,\n  "adapter_lr": 0.0001,\n  "head_lr": 0.001,\n  "weight_decay": 0.01,\n  "gradient_clip": 1.0,\n  "warmup_fraction": 0.1,\n  "dtype": "float16",\n  "max_overflow_retries": 3\n}\n'}
for name, source in BUNDLE.items():
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True, cwd=ROOT)


In [ ]:
# Find the official inputs. If ambiguous, set the two paths explicitly.
inputs = Path('/kaggle/input')
trains = [p for p in inputs.rglob('train.csv') if (p.parent / 'test.csv').is_file()]
bases = []
for path in inputs.rglob('config.json'):
    try:
        value = json.loads(path.read_text())
    except (ValueError, OSError):
        continue
    if (value.get('model_type'), value.get('hidden_size'), value.get('num_hidden_layers')) == ('qwen3', 4096, 36):
        bases.append(path.parent)
if len(trains) != 1 or len(bases) != 1:
    raise RuntimeError(f'Select one competition and one official Qwen3-8B Base: {trains}, {bases}')
TRAIN_CSV, BASE_MODEL = trains[0], bases[0]


In [ ]:
markers = list(Path('/kaggle/input').glob('**/own_lora24/final/COMPLETE.json'))
if len(markers) != 1:
    raise RuntimeError(f'Expected one own_lora24 checkpoint, found {markers}')
CHECKPOINT = markers[0].parent
run('scripts/validate_checkpoint.py', '--checkpoint', CHECKPOINT)
run('scripts/check_parity.py', '--checkpoint', CHECKPOINT, '--base-model', BASE_MODEL,
    '--train-csv', TRAIN_CSV, '--output', '/kaggle/working/parity.json')
run('scripts/predict.py', '--checkpoint', CHECKPOINT, '--base-model', BASE_MODEL,
    '--test-csv', TRAIN_CSV.parent / 'test.csv', '--output', '/kaggle/working/submission.parquet')
